# Ingest AEMO ST PASA forecast archives

Download historical forecast packages and extract their native 30-minute CSVs into Bronze. The Silver merge reads archive_uncompressed alongside the current forecasts. This notebook can run on its own schedule.

## 1. Discover historical packages

Match PUBLIC_STPASA_YYYYMMDD.zip in the archive listing. Keep the latest three months of packages.

In [ ]:
import os
import re
import time
import zipfile
from datetime import datetime
from urllib.parse import urljoin

import requests
from dateutil.relativedelta import relativedelta


archive_url = (
    "https://www.nemweb.com.au/"
    "Reports/ARCHIVE/Short_Term_PASA_Reports/"
)
def discover_zip_urls(folder, filename_pattern):
    # Read the live directory listing rather than constructing filenames.
    response = requests.get(folder, timeout=60)
    response.raise_for_status()

    files = re.findall(filename_pattern, response.text)

    return [
        urljoin(folder, filename)
        for filename in sorted(set(files))
    ]


archive_urls = discover_zip_urls(
    archive_url,
    r"PUBLIC_STPASA_\d{8}\.zip"
)

# Historical packages are large, so this implementation keeps only recent history.
start_date = datetime.today() - relativedelta(months=3)

archive_urls = [
    url
    for url in archive_urls
    if datetime.strptime(
        re.search(r"\d{8}", os.path.basename(url)).group(),
        "%Y%m%d"
    ) >= start_date
]

In [ ]:
print(f"Found {len(archive_urls)} historical packages")
archive_urls

## 2. Download missing archive ZIPs

Preserve source packages in Bronze. Skip files already downloaded.

In [ ]:
def download_if_not_exists(url, bronze_folder):
    filename = os.path.basename(url)
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url, timeout=60)

    response.raise_for_status()

    with open(path, "wb") as file:
        file.write(response.content)

    print(f"Finished: {filename} - {time.time() - start:.1f} seconds")

    return path

In [ ]:
archive_folder = (
    "/Volumes/workspace/default/aemo_mlops_volume/"
    "bronze/forecast_30min/archive"
)
os.makedirs(archive_folder, exist_ok=True)

for url in archive_urls:
    download_if_not_exists(url, archive_folder)


## 3. Extract historical forecasts

Each archive package contains individual forecast-run ZIPs. Extract their CSVs into archive_uncompressed and skip packages whose CSVs are already present.

In [ ]:
archive_csv_folder = archive_folder + "_uncompressed"

os.makedirs(archive_csv_folder, exist_ok=True)

archive_zip_files = [
    filename
    for filename in os.listdir(archive_folder)
    if filename.endswith(".zip")
]

start = time.time()
print(f"Found {len(archive_zip_files)} archive packages")

# First level: an archive package expands into individual forecast-run ZIPs.
for i, filename in enumerate(sorted(archive_zip_files), 1):
    path = os.path.join(archive_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        inner_zip_names = [
            member
            for member in zip_file.namelist()
            if member.lower().endswith(".zip")
        ]

        expected_csvs = [
            os.path.basename(os.path.splitext(member)[0] + ".CSV")
            for member in inner_zip_names
        ]

        already_extracted = expected_csvs and all(
            os.path.exists(os.path.join(archive_csv_folder, csv_name))
            for csv_name in expected_csvs
        )

        if already_extracted:
            print(f"[{i}/{len(archive_zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(archive_zip_files)}] Extracting: {filename}")
        zip_file.extractall(archive_csv_folder)


# Second level: each hourly ZIP expands into one STPASA CSV.
inner_zip_files = [
    filename
    for filename in os.listdir(archive_csv_folder)
    if filename.endswith(".zip")
]

print(f"Found {len(inner_zip_files)} inner ZIPs")

for i, filename in enumerate(sorted(inner_zip_files), 1):
    path = os.path.join(archive_csv_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        members = zip_file.namelist()

        already_extracted = all(
            os.path.exists(
                os.path.join(archive_csv_folder, os.path.basename(member))
            )
            for member in members
        )

        if not already_extracted:
            zip_file.extractall(archive_csv_folder)

    # Keep the final folder directly readable by Spark by removing inner ZIPs.
    os.remove(path)
    print(f"[{i}/{len(inner_zip_files)}] Extracted: {filename}")

print(f"Finished archive extraction in {time.time() - start:.1f} seconds")